# Fine-tuned main agent on Colab (E3)

The untuned **Qwen2.5-7B-Instruct** scored building macro F1 0.57 as the main agent (zero-shot and
few-shot, 336 test buildings): it saw the failing values but did not trace whether they block a room.
Here it learns the job with LoRA: input = the three GNN agents' findings (the same prompt as at run
time), target = the labels as an answer whose explanation traces the path first.
Training data: train + validation buildings only (`python -m src.agents.main_agent_sft` on the PC).

**Before you start**
1. *Runtime > Change runtime type*: **A100** (or L4: 4-bit, slower).
2. The Colab secret **`GH_TOKEN`** (as for the other notebooks).
3. Upload `D:\ada-llm-gnn\data\llm\main_agent_sft.jsonl` to **`MyDrive/ada-llm-gnn/data/llm/`**.
   The split, the agents and the IFC files are already on Drive from the E3 run.

Results: `MyDrive/ada-llm-gnn/experiments/<time>_main_agent/` (training) and
`<time>_eval_gnn_local_qwen-main_paths/` (the 336 test buildings).

*Status: written and checked offline only; not run on a GPU yet.*

In [ ]:
import subprocess
import torch

print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout)
VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 2**30
assert VRAM_GB >= 22, "pick an L4 or an A100 GPU" 

In [ ]:
import os
import subprocess

from google.colab import drive, userdata

drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/ada-llm-gnn"
REPO = "/content/ada-llm-gnn"
if not os.path.exists(REPO):
    url = f"https://{userdata.get('GH_TOKEN')}@github.com/Byomer1021/ada-llm-gnn.git"
    r = subprocess.run(["git", "clone", "-q", url, REPO], capture_output=True, text=True)
    if r.returncode:  # never print the URL: it contains the token
        raise RuntimeError("git clone failed: check the GH_TOKEN secret")
os.chdir(REPO)
subprocess.run(["git", "pull", "-q"], check=True)
for d in ("data", "models", "experiments"):  # the repo's output folders -> Drive
    os.makedirs(f"{DRIVE}/{d}", exist_ok=True)
    if not os.path.islink(d):
        subprocess.run(["rm", "-rf", d], check=True)
        os.symlink(f"{DRIVE}/{d}", d)
need = ["data/llm/main_agent_sft.jsonl", "data/datasets/split.json", "data/llm/teacher_verified.jsonl",
        "data/llm/student_verified.jsonl", "models/door.pt", "models/room.pt", "models/route.pt"]
missing = [p for p in need if not os.path.exists(p)]
assert not missing, f"missing on Drive: {missing}"
print(sum(1 for _ in open(need[0])), "training examples |",
      subprocess.run(["git", "log", "--format=%h %s", "-1"], capture_output=True, text=True).stdout)

## 1. Train (LoRA)

Settings: `config/main_agent_sft.yaml` (1 epoch, effective batch 16). A100 80 GB: 16-bit; smaller: 4-bit.
pip may print dependency conflicts (cudf, gradio, ...): those packages are not used, ignore them.

In [ ]:
!pip install -q -r requirements-student.txt
!pip uninstall -q -y torchao  # Colab's torchao 0.10 makes peft refuse to load

In [ ]:
flags = "--no-4bit" if VRAM_GB >= 70 else ""
print("16-bit LoRA" if flags else "4-bit QLoRA")
!python -m src.llm.train_student --config config/main_agent_sft.yaml {flags}

In [ ]:
import glob

ADAPTER = os.path.realpath(sorted(glob.glob("models/main_agent/*/adapter"))[-1])
print("adapter:", ADAPTER)

## 2. Serve the base model + the adapter (vLLM)

Installed after training (vLLM brings its own torch / transformers). Log: `/content/vllm.log`.

In [ ]:
!pip install -q vllm
!pip install -q ifcopenshell torch_geometric openai pyyaml scikit-learn networkx

In [ ]:
import json
import time
import urllib.request

import yaml

lora_r = yaml.safe_load(open("config/main_agent_sft.yaml"))["lora"]["r"]
log = open("/content/vllm.log", "w")
server = subprocess.Popen(
    ["vllm", "serve", "Qwen/Qwen2.5-7B-Instruct", "--enable-lora", "--max-lora-rank", str(lora_r),
     "--lora-modules", f"qwen-main={ADAPTER}", "--max-model-len", "8192",
     "--gpu-memory-utilization", "0.85", "--port", "8000"],
    stdout=log, stderr=subprocess.STDOUT)
for _ in range(240):
    try:
        models = json.load(urllib.request.urlopen("http://localhost:8000/v1/models", timeout=2))
        print("ready:", [m["id"] for m in models["data"]])
        break
    except Exception:
        if server.poll() is not None:
            raise RuntimeError("vLLM stopped:\n" + open("/content/vllm.log").read()[-3000:])
        time.sleep(5)
else:
    raise TimeoutError("vLLM not ready after 20 minutes, see /content/vllm.log")

## 3. Pilot: 10 test buildings

The untuned model said not_accessible on all 10. Look for `invalid` and `<-- wrong`.

In [ ]:
!python scripts/evaluate.py --provider local --llm-url http://localhost:8000/v1 --llm-model qwen-main --workers 8 --limit 10

## 4. E3: all 336 test buildings

Paste the printed metrics back. For comparison (same agents, same buildings): rule paths 0.893,
GPT-OSS 20B 0.899, Qwen 7B untuned 0.57.

In [ ]:
!python scripts/evaluate.py --provider local --llm-url http://localhost:8000/v1 --llm-model qwen-main --workers 16 | grep -E 'findings for|"(accuracy|macro_f1|invalid_answers|precision|recall|f1|exact_match|accessible|not_accessible)"|saved'

In [ ]:
server.terminate()  # then: Runtime > Disconnect and delete runtime